In [2]:
import os
import re
import time
import ee
import numpy as np
import pandas as pd

# ==============================================================================
# 1. إعداد المشروع والاتصال بـ Google Earth Engine
# ==============================================================================
# استبدل القيمة بالأسفل بمعرّف مشروعك الإنجليزي الجديد في حال استمرار restricted mode
PROJECT_ID = "acoustic-gizmo-513018-a8"

if not PROJECT_ID.isascii() or " " in PROJECT_ID:
    raise ValueError("يجب أن يكون PROJECT_ID باللغة الإنجليزية وبدون مسافات (مثال: field-shift-egypt-441512).")

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# ==============================================================================
# 2. تعريف المراكز الزراعية الـ 14 في مصر
# ==============================================================================
HUBS = [
    ("Kafr_ElSheikh", 31.11, 30.94), ("Dakahlia", 31.04, 31.38),
    ("Sharqia", 30.72, 31.67), ("Beheira", 31.03, 30.47),
    ("Gharbia", 30.79, 31.00), ("Menoufia", 30.55, 31.01),
    ("Qalyubia", 30.46, 31.18), ("Salhia_Reclaimed", 30.65, 31.95),
    ("Fayoum", 29.31, 30.84), ("Beni_Suef", 29.07, 31.09),
    ("Minya", 28.09, 30.75), ("Assiut", 27.18, 31.18),
    ("Sohag", 26.56, 31.69), ("Qena_Luxor", 25.69, 32.64),
]

hubs_fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([lon, lat]), {"Governorate": name})
    for name, lat, lon in HUBS
])

# ==============================================================================
# 3. تجهيز مجموعة بيانات الأمطار NASA GPM IMERG V07
# ==============================================================================
gpm_col = ee.ImageCollection("NASA/GPM_L3/IMERG_V07").select("precipitation")
BANDS = ["GPM_Precip_Mean_mm_hr", "GPM_Max_Storm_Intensity_mm_hr"]


def safe_stat(col, stat_type, band_name):
    dummy = ee.Image.constant([0]).toFloat().rename([band_name]).updateMask(ee.Image.constant(0))
    merged = col.map(lambda img: img.toFloat().rename([band_name])).merge(ee.ImageCollection([dummy]))
    return merged.mean() if stat_type == "mean" else merged.max()


def get_info_with_retry(ee_obj, retries=5):
    for attempt in range(retries):
        try:
            return ee_obj.getInfo()
        except Exception as err:
            if attempt == retries - 1:
                raise
            wait_s = 15 * (attempt + 1)
            print(f"      ⏳ ضغط مؤقت على الخادم ({err}) -> انتظار {wait_s} ثانية...", flush=True)
            time.sleep(wait_s)


# ==============================================================================
# 4. بناء الشبكة الزمنية (365 فترة × 10 أيام) والسحب على دفعات خفيفة
# ==============================================================================
date_bins = pd.date_range("2016-01-01", "2025-12-31", freq="10D")
expected_dates = pd.to_datetime(date_bins[:-1])
intervals = [
    (date_bins[i].strftime("%Y-%m-%d"), date_bins[i + 1].strftime("%Y-%m-%d"))
    for i in range(len(date_bins) - 1)
]

BATCH_SIZE = 18  # 6 أشهر لكل دفعة لمنع تجاوز الذاكرة أو المهلة
batches = [intervals[i:i + BATCH_SIZE] for i in range(0, len(intervals), BATCH_SIZE)]

records_dict = {}
print(f"📥 [4/6] بدء سحب بيانات NASA GPM IMERG لـ 14 محافظة ({len(batches)} دفعة)...", flush=True)

for idx, batch in enumerate(batches, start=1):
    imgs = []
    for s, e in batch:
        sub = gpm_col.filterDate(s, e)
        img = (
            safe_stat(sub, "mean", "GPM_Precip_Mean_mm_hr")
            .addBands(safe_stat(sub, "max", "GPM_Max_Storm_Intensity_mm_hr"))
            .rename([f"{s}_{b}" for b in BANDS])
        )
        imgs.append(img)

    fc_batch = ee.ImageCollection(imgs).toBands().reduceRegions(
        collection=hubs_fc,
        reducer=ee.Reducer.first(),
        scale=11132,
        tileScale=4
    )
    res = get_info_with_retry(fc_batch)

    for feat in res["features"]:
        props = feat["properties"]
        gov = props["Governorate"]
        for k, val in props.items():
            m = re.match(r"^(?:\d+_)?(\d{4}-\d{2}-\d{2})_(.+)$", k)
            if m:
                records_dict.setdefault((gov, m.group(1)), {})[m.group(2)] = val

    print(f"   ✓ GPM: اكتملت الدفعة {idx}/{len(batches)} ({batch[0][0]} إلى {batch[-1][1]})", flush=True)
    time.sleep(1.5)

# ==============================================================================
# 5. المطابقة على الشبكة الموحدة وتنظيف القيم وحفظ الملف
# ==============================================================================
df_raw = pd.DataFrame([{"Governorate": gov, "Date": dt, **vals} for (gov, dt), vals in records_dict.items()])
df_raw["Date"] = pd.to_datetime(df_raw["Date"])

grid = pd.MultiIndex.from_product(
    [[h[0] for h in HUBS], expected_dates],
    names=["Governorate", "Date"]
).to_frame().reset_index(drop=True)

out = pd.merge(grid, df_raw, on=["Governorate", "Date"], how="left").sort_values(["Governorate", "Date"]).reset_index(drop=True)

num_cols = out.select_dtypes(include=[np.number]).columns
out[num_cols] = out.groupby("Governorate")[num_cols].transform(lambda g: g.interpolate("linear").ffill().bfill())
out["Date"] = out["Date"].dt.strftime("%Y-%m-%d")

output_path = r"d:\vscode\NASA\DATA\04_GPM_Precipitation.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
out.to_csv(output_path, index=False)

print(f"\n✅ اكتمل الملف الرابع بنجاح: {output_path}")
print(f"📐 الأبعاد: {out.shape[0]} صف × {out.shape[1]} عمود | القيم المفقودة: {out.isnull().sum().sum()}")

C:\Users\eng mohamed\AppData\Roaming\Python\Python314\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


📥 [4/6] بدء سحب بيانات NASA GPM IMERG لـ 14 محافظة (21 دفعة)...
   ✓ GPM: اكتملت الدفعة 1/21 (2016-01-01 إلى 2016-06-29)


C:\Users\eng mohamed\AppData\Roaming\Python\Python314\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


   ✓ GPM: اكتملت الدفعة 2/21 (2016-06-29 إلى 2016-12-26)
   ✓ GPM: اكتملت الدفعة 3/21 (2016-12-26 إلى 2017-06-24)
   ✓ GPM: اكتملت الدفعة 4/21 (2017-06-24 إلى 2017-12-21)
   ✓ GPM: اكتملت الدفعة 5/21 (2017-12-21 إلى 2018-06-19)
   ✓ GPM: اكتملت الدفعة 6/21 (2018-06-19 إلى 2018-12-16)
   ✓ GPM: اكتملت الدفعة 7/21 (2018-12-16 إلى 2019-06-14)
   ✓ GPM: اكتملت الدفعة 8/21 (2019-06-14 إلى 2019-12-11)
   ✓ GPM: اكتملت الدفعة 9/21 (2019-12-11 إلى 2020-06-08)
   ✓ GPM: اكتملت الدفعة 10/21 (2020-06-08 إلى 2020-12-05)
   ✓ GPM: اكتملت الدفعة 11/21 (2020-12-05 إلى 2021-06-03)
   ✓ GPM: اكتملت الدفعة 12/21 (2021-06-03 إلى 2021-11-30)
   ✓ GPM: اكتملت الدفعة 13/21 (2021-11-30 إلى 2022-05-29)
   ✓ GPM: اكتملت الدفعة 14/21 (2022-05-29 إلى 2022-11-25)
   ✓ GPM: اكتملت الدفعة 15/21 (2022-11-25 إلى 2023-05-24)
   ✓ GPM: اكتملت الدفعة 16/21 (2023-05-24 إلى 2023-11-20)
   ✓ GPM: اكتملت الدفعة 17/21 (2023-11-20 إلى 2024-05-18)
   ✓ GPM: اكتملت الدفعة 18/21 (2024-05-18 إلى 2024-11-14)
   ✓ GPM: اكتملت الدفع